# 5 - Single-Variable Regressions

For each X, fit `life_expectancy ~ X` and look at the scatter, regression line, slope, p-value, and R-squared. These tell us about marginal correlation only - they don't control for anything else.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
df = pd.read_csv('data/merged_df.csv', dtype={'fips': str}).dropna(
    subset=['life_expectancy','median_hh_income','pct_bachelors_plus',
            'pct_poverty','pct_nhwhite','median_age','log_population']
)
print('Analysis sample:', len(df), 'counties')
df.head()

### Results table - slope, p-value, R-squared for each predictor

In [ ]:
predictors = ['median_hh_income','pct_bachelors_plus','pct_poverty',
              'pct_nhwhite','median_age','log_population']

rows = []
for p in predictors:
    X = sm.add_constant(df[p])
    m = sm.OLS(df['life_expectancy'], X).fit()
    rows.append({'predictor': p,
                 'coef':  round(m.params[p], 4),
                 'p_value': m.pvalues[p],
                 'R2':    round(m.rsquared, 3)})
results = pd.DataFrame(rows).sort_values('R2', ascending=False)
results

### Scatterplots with regression lines

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.flatten(), predictors):
    sns.regplot(data=df, x=p, y='life_expectancy', ax=ax,
                scatter_kws={'alpha':0.25, 's':12, 'color':'#3b7dd8'},
                line_kws={'color':'#d8453b'})
    X = sm.add_constant(df[p])
    m = sm.OLS(df['life_expectancy'], X).fit()
    ax.set_title(f'{p}\nR2={m.rsquared:.2f}, p={m.pvalues[p]:.1e}', fontsize=10)
    ax.set_ylabel('Life expectancy (yrs)')
plt.tight_layout(); plt.show()

### Closer look at the strongest single predictor

In [ ]:
strongest = results.iloc[0]['predictor']
print('Strongest single predictor:', strongest)
X = sm.add_constant(df[strongest])
m = sm.OLS(df['life_expectancy'], X).fit()
print(m.summary())

**What to expect:** Median household income and % bachelor's+ should each individually explain 30-50% of the variance in county life expectancy. Poverty rate will be similar (it's negatively correlated with the first two). Race composition (`pct_nhwhite`) will show a positive slope nationally - but that's largely a confound with regional poverty patterns, which we'll see dissolve in the multivariable model.

**Next:** notebook 6 - combine the predictors.